# 歌词采集-QQ音乐
不需要按专辑采集

In [1]:
import json
import pandas as pd

import os

In [2]:
import sys

sys.path.append('..')

from data_crawler import  get_songs_data_raw, get_all_songs_lyric, clear_and_save_lyric
from songs_libs import format_timestamp, SongDataCleaner

In [3]:
albums_to_delete = ['声生不息', '我歌', '中国梦', '谁是大歌神', '梦想的声音', '我是歌手', 'JJ的咖啡调调', '不凡的改变', '“17聚幸福”江苏卫视2017跨年演唱会', '江苏卫视', '湖南卫视', '浙江卫视', '启航2020', '2018中国蓝', '时光音乐会', '经典咏流传', '天籁', '剧好听的歌']

# 批量数据采集

In [ ]:
singers = [('luodayou', '罗大佑'), ('lizongsheng', '李宗盛'), ('zhangxueyou', '张学友'), ('twins', 'Twins'), ('wangsulong', '汪苏泷'), ('panweibo', '潘玮柏'), ('dengziqi', 'G.E.M. 邓紫棋'), ('xuezhiqian', '薛之谦'), ('xusong', '许嵩'), ('zhangjie', '张杰'), ('taozhe', '陶喆'), ('fangdatong', '方大同'), ('wangfei', '王菲'), ('maobuyi', '毛不易'), ('beyond', 'BEYOND')]
max_page = 10

for i in singers[-1:]:
    file_path_prefix = f"data/{i[0]}/"
    # 如果file_path_prefix不存在，则新建
    if not os.path.exists(file_path_prefix):
        os.makedirs(file_path_prefix)
    # 原始曲目数据采集
    song_data_raw = get_songs_data_raw(singger=i[0], max_page=max_page)
    df_song_data_raw = pd.DataFrame(song_data_raw)
    # 原始曲目保存
    df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)
    # 重新读取数据
    df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
    song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')
    # 删除live歌曲
    df_songs = SongDataCleaner.clear_live_songs(df_song_data_raw_read)
    # 歌曲名清洗
    df_songs = SongDataCleaner.clear_song_name(df_songs)
    # 仅含歌手独唱歌曲
    df_songs = SongDataCleaner.clear_song_singer(df_songs, i[1])
    # 删除晚会歌曲
    df_songs = SongDataCleaner.clear_song_tv_show(df_songs, albums_to_delete)
    df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first').reset_index(drop=True)

    df_songs_final = df_songs.head(130)
    df_songs_final['publish_date'] = df_songs_final['publish_time'].apply(
        lambda x: format_timestamp(x))
    df_songs_final = df_songs_final[df_songs_final['publish_date'].str.contains('-')]
    df_songs_final['publish_year'] = df_songs_final['publish_date'].apply(
        lambda x: x.split('-')[0])
    df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)
    # 歌词采集与清洗
    get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', df_songs_final)
    clear_and_save_lyric(file_path_prefix, df_songs_final)

# main

In [4]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

# file_path_prefix = "data/liyuchun/"
# singer = "李宇春"
# max_page = 14

file_path_prefix = "data/xuezhiqian/"
singer = "薛之谦"
max_page = 10
if not os.path.exists(file_path_prefix):
    os.makedirs(file_path_prefix)

### 曲目采集

In [5]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singer, max_page=max_page)

正在获取第1页数据...
正在获取第2页数据...
正在获取第3页数据...
正在获取第4页数据...
正在获取第5页数据...
正在获取第6页数据...
正在获取第7页数据...
正在获取第8页数据...
正在获取第9页数据...
正在获取第10页数据...


In [6]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [7]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')
df_song_data_raw_read

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time
0,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,318,1775750401
1,102636799,001Qu4I30eVFYb,演员,NaN,薛之谦,5062,002J4UUk29y8BY,绅士,989994,003y8dsH2wBHlo,261,1433433600
2,104775877,003ouHMP12glVD,其实,《妈妈像花儿一样》电视剧插曲,薛之谦,5062,002J4UUk29y8BY,意外,443691,000QgFcm0v8WaF,242,1384099200
3,272125057,0013WPvt4fQH2b,天外来物,NaN,薛之谦,5062,002J4UUk29y8BY,天外来物,16596032,000K9Zp13TZp5s,257,1609344000
4,233704383,002zfxmN2e1vLQ,陪你去流浪,NaN,薛之谦,5062,002J4UUk29y8BY,尘,7064087,000DMpJ73yeITP,274,1577376000
...,...,...,...,...,...,...,...,...,...,...,...,...
295,108486069,0034JD0U2dzw5K,我好像在哪见过你 (Live),NaN,Adinda;薛之谦,1186071,004UGnzU27oiV1,中国新声代第四季 第11期,1600784,004UIVfr2Rtb9d,119,1473436800
296,127553012,0032JCSD13qDb0,守候 (2020重唱版),NaN,薛之谦,5062,002J4UUk29y8BY,一首歌一个故事,14986328,003dXfPM4PNl0C,292,1600272000
297,609444698,0027GHQO0sI5H6,薛之谦《丑八怪》,NaN,是文川吖,0,0032fmHO2UDnV3,人生起起落落,53654363,003UEvP54QLbxW,261,1760025600
298,609467688,000NiOrs05ncM9,薛之谦《无数 (2023百川狂想曲现场) 》,NaN,海山来了,0,0032fmHO2UDnV3,时光印记,53654378,002XFogy3zafLn,197,1760025600


### 清洗

In [8]:
# 删除live歌曲
df_songs = SongDataCleaner.clear_live_songs(df_song_data_raw_read)
# df_songs = df_song_data_raw_read[~df_song_data_raw_read['song_name'].str.contains('口白')]
df_songs = df_songs[~df_songs['song_name'].str.contains('现场版', case=False)]
df_songs = df_songs[~df_songs['song_name'].str.contains('&', case=False)]
df_songs = df_songs[~df_songs['song_name'].str.contains('【', case=False)]
df_songs = df_songs[~df_songs['song_name'].str.contains('／', case=False)]

# 歌曲名清洗
df_songs = SongDataCleaner.clear_song_name(df_songs)
# 仅含歌手独唱歌曲
df_songs = SongDataCleaner.clear_song_singer(df_songs, singer)
# 删除晚会歌曲
df_songs = SongDataCleaner.clear_song_tv_show(df_songs, albums_to_delete)
df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first').reset_index(drop=True)
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure
0,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,318,1775750401,顽疾,顽疾,顽疾
1,102636799,001Qu4I30eVFYb,演员,NaN,薛之谦,5062,002J4UUk29y8BY,绅士,989994,003y8dsH2wBHlo,261,1433433600,演员,演员,绅士
2,104775877,003ouHMP12glVD,其实,《妈妈像花儿一样》电视剧插曲,薛之谦,5062,002J4UUk29y8BY,意外,443691,000QgFcm0v8WaF,242,1384099200,其实,其实,意外
3,272125057,0013WPvt4fQH2b,天外来物,NaN,薛之谦,5062,002J4UUk29y8BY,天外来物,16596032,000K9Zp13TZp5s,257,1609344000,天外来物,天外来物,天外来物
4,233704383,002zfxmN2e1vLQ,陪你去流浪,NaN,薛之谦,5062,002J4UUk29y8BY,尘,7064087,000DMpJ73yeITP,274,1577376000,陪你去流浪,陪你去流浪,尘
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,647958,002teWyP3uBNgN,马戏小丑,NaN,薛之谦,5062,002J4UUk29y8BY,未完成的歌,55085,0036fT613sAeZn,303,1259596800,马戏小丑,马戏小丑,未完成的歌
111,650356745,004W4x5G4Kjpen,粉钻 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,250,1773936000,粉钻,粉钻,nan
112,1333806,000fjX1m12reIT,快乐帮,NaN,薛之谦,5062,002J4UUk29y8BY,薛之谦,51504,003mUYW22JXKVK,243,1136044800,快乐帮,快乐帮,薛之谦
113,649823961,003Rpde21bLvWc,造物 (2026万兽之王世界巡回演唱会彩排版),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,64,1773763200,造物,造物,nan


In [9]:
# 特别处理
df_songs['album_name'] = df_songs['album_name'].fillna('').astype(str)
df_songs = df_songs[~df_songs['album_name'].str.contains('Unforgettable')]
# 专辑名包含'爱如此神奇'，但保留song_name包含'中国人'的行
df_songs = df_songs[~df_songs['album_name'].str.contains('爱如此神奇') | df_songs['song_name'].str.contains('中国人')]
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure
0,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,318,1775750401,顽疾,顽疾,顽疾
1,102636799,001Qu4I30eVFYb,演员,NaN,薛之谦,5062,002J4UUk29y8BY,绅士,989994,003y8dsH2wBHlo,261,1433433600,演员,演员,绅士
2,104775877,003ouHMP12glVD,其实,《妈妈像花儿一样》电视剧插曲,薛之谦,5062,002J4UUk29y8BY,意外,443691,000QgFcm0v8WaF,242,1384099200,其实,其实,意外
3,272125057,0013WPvt4fQH2b,天外来物,NaN,薛之谦,5062,002J4UUk29y8BY,天外来物,16596032,000K9Zp13TZp5s,257,1609344000,天外来物,天外来物,天外来物
4,233704383,002zfxmN2e1vLQ,陪你去流浪,NaN,薛之谦,5062,002J4UUk29y8BY,尘,7064087,000DMpJ73yeITP,274,1577376000,陪你去流浪,陪你去流浪,尘
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,647958,002teWyP3uBNgN,马戏小丑,NaN,薛之谦,5062,002J4UUk29y8BY,未完成的歌,55085,0036fT613sAeZn,303,1259596800,马戏小丑,马戏小丑,未完成的歌
111,650356745,004W4x5G4Kjpen,粉钻 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,,0,NaN,250,1773936000,粉钻,粉钻,nan
112,1333806,000fjX1m12reIT,快乐帮,NaN,薛之谦,5062,002J4UUk29y8BY,薛之谦,51504,003mUYW22JXKVK,243,1136044800,快乐帮,快乐帮,薛之谦
113,649823961,003Rpde21bLvWc,造物 (2026万兽之王世界巡回演唱会彩排版),NaN,薛之谦,0,0032fmHO2UDnV3,,0,NaN,64,1773763200,造物,造物,nan


In [10]:
# 前130首
songs_list_all = df_songs['song_name_pure'].to_list()
songs_list_130 = df_songs.head(130)['song_name_pure'].to_list()

## 歌单确认

In [11]:
# 李宇春
songs_to_add = [
    '皇后与梦想', '下雨', '冰菊物语', '我的王国', '漂浮地铁', '今天有朵云爱我', '您所拨打的电话号码是空号',
    '一而再再而三地喜欢你', '人间乐园', 'TMD我爱你', '口音', '木兰', '开放'
]
songs_to_delete = ['今夜你会不会来', '春风十里', '情书', '那女孩对我说', '南方姑娘', '爱你所爱', '无心睡眠', '莫过于此', '天黑黑', '张三的歌', '漂洋过海来看你', '城里的月光', '不要对他说', '下个,路口,见']
# 陈奕迅
songs_to_delete = ['新曲+精选', 'K歌之王AIR', '慢慢喜欢你', '最冷一天']
# 任贤齐
songs_to_delete = ['伤心太平洋+心太软+我是一只鱼+对面的女孩看过来', '桥边姑娘', '你知道我在等你吗', '我是一只小小鸟', '外婆的澎湖湾2015', '海阔天空', '爱的路上只有你和我', '爱的初体验', '美丽的坏女人', '朋友的酒']
# 林俊杰
# songs_to_delete = ['开场白', '无聊', '起风了']
# 陈信宏
# songs_to_delete = ['玫瑰少年-FromTHEFIRSTTAKE', '疯狂世界+候鸟', '2010离开地球表面', '盛夏光年×HIPHOPMAN', 'Paradise+倔强', 'Hosee', 'SHERO']
# 蔡依林
# songs_to_delete = ['看我七十二变', '爱情36计']
# 伍佰
# songs_to_delete = ['少年吔,安啦！']
# 凤凰传奇
songs_to_delete = [
    '海底', 'mrs.leta', '好运来', '普通disco', '【拜新年】专辑歌曲串烧', '好汉歌', '狼的诱惑广场舞版',
    '专辑歌曲串烧', '歌曲串烧', 'dj天籁传奇', '天蓝蓝)'
]
# Beyond
songs_to_delete = [
    '为了你,为了我', '真的爱妳', '遥かなる夢に', '灰色軌跡', '遥かなる梦に〜Faraway〜',
    'リゾ·ラバ～International～', 'Cryin', '遥かなるゆめに～Faraway', '喜欢妳'
]

# 罗大佑
songs_to_delete = ['童年full']

# 方大同
songs_to_delete = ['假行僧', '金砖的秘密', '月亮代表我的心']

# 李宗盛
# songs_to_delete = ['我听见有人叫你宝贝', '17岁女生的温柔', '漂洋过海来看你']

# 莫文蔚
# songs_to_delete = ['当你老了', '夜上海']

# 毛不易
songs_to_add = ['消愁']
songs_to_delete = ['小王日记']

# 刘德华
songs_to_add = ['中国人']
songs_to_delete = ['再吻我吧!', '又弹起心爱的土琵琶']

songs_list_final = songs_list_130.copy()

In [ ]:
# 添加
for i in songs_to_add:
    if i not in songs_list_final[:100]:
        print(i)
        # 添加到列表的第一个
        songs_list_final.insert(0, i)

In [ ]:
# 删除
for i in songs_to_delete:
    if i in songs_list_final:
        print(i)
        songs_list_final.remove(i)

In [12]:
len(songs_list_final)

115

## 歌曲数据确认

In [13]:
df_songs_final = df_songs[df_songs['song_name_pure'].isin(
    songs_list_final)].reset_index(drop=True)

# df_songs_final = df_songs_final.drop(columns=['song_name_unique'])
# df_songs_final['song_name_unique'] = df_songs_final['song_name_pure']
df_songs_final['publish_date'] = df_songs_final['publish_time'].apply(
    lambda x: format_timestamp(x))
df_songs_final = df_songs_final[df_songs_final['publish_date'].str.contains('-')]
df_songs_final['publish_year'] = df_songs_final['publish_date'].apply(
    lambda x: x.split('-')[0])
df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,318,1775750401,顽疾,顽疾,顽疾,2026-04-10,2026
1,102636799,001Qu4I30eVFYb,演员,NaN,薛之谦,5062,002J4UUk29y8BY,绅士,989994,003y8dsH2wBHlo,261,1433433600,演员,演员,绅士,2015-06-05,2015
2,104775877,003ouHMP12glVD,其实,《妈妈像花儿一样》电视剧插曲,薛之谦,5062,002J4UUk29y8BY,意外,443691,000QgFcm0v8WaF,242,1384099200,其实,其实,意外,2013-11-11,2013
3,272125057,0013WPvt4fQH2b,天外来物,NaN,薛之谦,5062,002J4UUk29y8BY,天外来物,16596032,000K9Zp13TZp5s,257,1609344000,天外来物,天外来物,天外来物,2020-12-31,2020
4,233704383,002zfxmN2e1vLQ,陪你去流浪,NaN,薛之谦,5062,002J4UUk29y8BY,尘,7064087,000DMpJ73yeITP,274,1577376000,陪你去流浪,陪你去流浪,尘,2019-12-27,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,647958,002teWyP3uBNgN,马戏小丑,NaN,薛之谦,5062,002J4UUk29y8BY,未完成的歌,55085,0036fT613sAeZn,303,1259596800,马戏小丑,马戏小丑,未完成的歌,2009-12-01,2009
111,650356745,004W4x5G4Kjpen,粉钻 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,,0,NaN,250,1773936000,粉钻,粉钻,nan,2026-03-20,2026
112,1333806,000fjX1m12reIT,快乐帮,NaN,薛之谦,5062,002J4UUk29y8BY,薛之谦,51504,003mUYW22JXKVK,243,1136044800,快乐帮,快乐帮,薛之谦,2006-01-01,2006
113,649823961,003Rpde21bLvWc,造物 (2026万兽之王世界巡回演唱会彩排版),NaN,薛之谦,0,0032fmHO2UDnV3,,0,NaN,64,1773763200,造物,造物,nan,2026-03-18,2026


In [14]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

## 歌词采集

In [15]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', df_songs_final)

顽疾
粉钻 (2026万兽之王巡回演唱会广州站现场)
造物 (2026万兽之王世界巡回演唱会彩排版)
平庸 (2026万兽之王巡回演唱会广州站现场)


In [16]:
# 读取raw_lyric_data.json
with open(file_path_prefix + 'raw_lyric_data.json', 'r', encoding='utf-8') as f:
    lyric_raw = json.load(f)
# 删除lyric_raw中lyric_raw为空的元素
lyric_raw = [song for song in lyric_raw if song['lyric_raw']]

# 替换raw_lyric_data.json
with open(file_path_prefix + 'raw_lyric_data.json', 'w', encoding='utf-8') as f:
    json.dump(lyric_raw, f, ensure_ascii=False, indent=4)

len(lyric_raw)

# 重新运行上一个cell

115

## 歌词清洗

In [17]:
clear_and_save_lyric(file_path_prefix, df_songs_final)

In [18]:
# 歌词数据查验
df_lyric = pd.read_json(f"{file_path_prefix}cleared_lyric_data.json")
df_lyric['lyric_length'] = df_lyric['lyrics_text'].apply(lambda x: len(x))
df_lyric.sort_values(by='lyric_length')

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text,lyric_length
113,649823961,造物 (2026万兽之王世界巡回演唱会彩排版),2026-04-11 00:03:36,1,李聪/羽田,羽田,,熟睡之前。花朵是，温驯的妖魔。吞咽之后。喷涌十万，异化的触手。已经都失控了。反噬，腐朽。你何...,107
52,209213590,渡,2026-04-11 00:45:36,1,薛之谦,Torbjorn Brundtland/Svein Berge,郑伟,渡人去的夜，用稀有的火焰。照亮了胆怯，燃尽我语言。亏欠都是磁铁，也不能被降解。都想赎去罪孽，...,160
70,290870320,把你揉碎捏成苹果,2026-04-11 00:14:06,1,方毅,方毅,方毅,别再从我身边走过。这里已经是片沙漠。没有欲望让你解渴。怪谁呢。我就当你从没来过。渐渐掉入这个...,206
86,233704387,环,2026-04-11 00:24:42,1,孟楠,孟楠,郑伟,琉璃墙后面，玲珑的脸。未尝尽波澜，却想靠岸。不是人卑贱，是门虚掩。迷茫的试探，找答案。美若天...,211
111,650356745,粉钻 (2026万兽之王巡回演唱会广州站现场),2026-04-11 00:07:06,1,薛之谦,薛之谦,,满地粉钻，无人看管。你若不甘，用挚爱交换。漫天红伞，无人生还。我的遗憾，是不能洁白的带你离开...,245
...,...,...,...,...,...,...,...,...,...
55,233704309,慢半拍,2026-04-11 00:18:07,1,薛之谦,许嵩,许嵩,对过往的自己，敬个礼。从此，再伤害我也没关系。接受过蚂蚁，簇拥过华丽。谁还敢逃避。浓妆艳抹，...,609
62,406405348,Nothing,2026-04-11 00:17:35,1,郭冠廷,薛之谦,周以力,Guess，I，should，leave。The，train，is，here，on，time...,633
99,1771030,为什么,2026-04-11 00:20:52,1,薛之谦,薛之谦,,要赶上六点半的早操。背上我最爱的书包往学校跑，快跑。六十块的校服一套。红领巾要戴出领带的味道...,657
106,1257203,星河之役,2026-04-11 00:23:23,1,陈耀川,Garden,,是从哪里，几光年的距离。银河的边际。传来求救的讯息。A，o，a，o，a。Ooh。是狼牙星，的...,892


In [ ]:
# 歌词文本长度大于40
df_lyric = df_lyric[df_lyric['lyric_length'] > 220]
df_lyric.sort_values(by='lyric_length')

In [ ]:
# 删除有问题的
song_ids_to_delete = [404796564]
df_lyric = df_lyric[~df_lyric['song_id'].isin(song_ids_to_delete)]
df_lyric

In [ ]:
# 覆盖原文件
df_lyric['start_time'] = df_lyric['start_time'].astype(str).apply(lambda x: x.split(' ')[1])
df_lyric_d = df_lyric.to_dict(orient='records')
with open(file_path_prefix + 'cleared_lyric_data.json', 'w',
              encoding='utf-8') as f:
        json.dump(df_lyric_d, f, ensure_ascii=False, indent=4)

## 特别处理

In [ ]:
df_lyric['lyricist'].unique()

In [ ]:
lyricist = ['阿信', '五月天阿信', '五月天 阿信', '阿信(五月天)']
df_lyric_flited = df_lyric[df_lyric['lyricist'].isin(lyricist)]
df_lyric_flited

In [ ]:
songs_to_delete_lyric = ['派对动物 + 离开地球表面 (live in the sky)', '伤心的人就听撑腰 (Life Live)', '明白 (后段) (Live)']
df_lyric_flited = df_lyric_flited[~df_lyric_flited['song_name'].isin(songs_to_delete_lyric)]
df_lyric_flited

In [ ]:
# 将df_lyric_flited保存为json
df_lyric_flited = df_lyric_flited.copy()
df_lyric_flited['start_time'] = df_lyric_flited['start_time'].astype(str).apply(lambda x: x.split(' ')[1])
df_lyric_flited_d = df_lyric_flited.to_dict(orient='records')
with open(file_path_prefix + 'cleared_lyric_data.json', 'w',
              encoding='utf-8') as f:
        json.dump(df_lyric_flited_d, f, ensure_ascii=False, indent=4)